# Tutorial 03: Pozos Cuánticos Finitos, Doble Pozo y Efecto Klein (`ScalarPotential`)

La clase `ScalarPotential(func)` permite acoplar cualquier función espacial definida en Python $V(\mathbf{r})$ directamente al motor C++ de `dirac_solver`.

En mecánica cuántica relativista, cuando una partícula de Dirac con energía $E = \sqrt{p^2 + m^2}$ incide sobre una barrera electrostática de altura $V_0 > E + m$, la solución dentro de la barrera no decae exponencialmente sino que **oscila** al acoplarse con el continuo de energía negativa (estados de positrón/hueco). Este fenómeno puramente relativista se conoce como **Tunelamiento de Klein (Paradoja de Klein)**.

### Objetivos de aprendizaje
1. Construir potenciales personalizados con `ScalarPotential(func)`: pozo finito suave, doble pozo simétrico y barrera de potencial relativista.
2. Simular un paquete de ondas atrapado parcialmente en un pozo finito y en un doble pozo cuántico.
3. Simular la colisión contra una barrera supercrítica ($V_0 > E + m$) y cuantificar las probabilidades de reflexión ($R$) y transmisión de Klein ($T$).

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Asegurar que dirac_solver sea localizable desde examples/, examples/es/ o examples/en/
for _rel in ['..', '../..']:
    _abs = os.path.abspath(_rel)
    if os.path.exists(os.path.join(_abs, 'dirac_solver')) and _abs not in sys.path:
        sys.path.insert(0, _abs)

from dirac_solver import (
    Grid,
    ConstantSpinor,
    GaussianPacket,
    PlaneWave,
    DiracProblemBuilder,
    DiracSolver,
    FreeParticle,
    ScalarPotential,
    CoulombPotential,
    YukawaPotential,
    InfiniteWellPotential,
    Observables,
    HDF5Storage,
    _core,
    electron_mass,
    c,
    hbar,
)
from dirac_solver.storage import create_animation, create_animation_1d

plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 12})

---
## 1. Pozo Cuántico Finito y Doble Pozo Simétrico con `ScalarPotential`

Definiremos dos perfiles clásicos mediante funciones de Python que reciben la lista de coordenadas `r = [x, y, z]`:
1. **Pozo finito cuadrado suavizado**: $V(x) = 0$ para $|x| < 4$, y $V(x) = V_0 = 2.5$ fuera.
2. **Doble pozo cuántico**: dos mínimos en $x = \pm 3.0$ separados por una barrera central en $x = 0$.

In [ ]:
grid_1d = Grid(shape=(400,), spacing=(0.08,))
x = grid_1d.coords[:, 0]
dV = grid_1d.get_cell_volume()

def finite_well_func(r):
    pos_x = r[0]
    if -4.0 < pos_x < 4.0:
        return 0.0
    return 2.5

def double_well_func(r):
    pos_x = r[0]
    # Dos pozos en [-5, -1] y [1, 5] con una barrera central en [-1, 1]
    if -5.0 < pos_x < -1.0 or 1.0 < pos_x < 5.0:
        return 0.0
    elif -1.0 <= pos_x <= 1.0:
        return 2.0
    return 3.5

pot_finite_well = ScalarPotential(finite_well_func)
pot_double_well = ScalarPotential(double_well_func)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, [pot_finite_well.evaluate([xi]) for xi in x], 'b-', lw=2, label='Pozo Finito')
ax.plot(x, [pot_double_well.evaluate([xi]) for xi in x], 'r--', lw=2, label='Doble Pozo')
ax.set_xlabel('Posición x')
ax.set_ylabel('Potencial V(x)')
ax.set_title('Perfiles de Potencial Personalizados con ScalarPotential')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Simulación en el Doble Pozo Cuántico: paquete lanzado desde el pozo izquierdo hacia la barrera central
spinor_dw = ConstantSpinor(momentum=[2.2, 0.0, 0.0])
initial_dw = GaussianPacket(spinor_dw, center=[-3.0, 0.0, 0.0], spatial_width=0.7)

problem_dw = (
    DiracProblemBuilder()
    .set_grid(grid_1d)
    .set_initial_state(initial_dw)
    .set_potential(pot_double_well)
    .set_boundary_condition(_core.AbsorbingBoundary(strength=0.3, width=30))
    .set_time_parameters(time_step=0.004, total_time=3.5)
    .build()
)

solver_dw = DiracSolver(problem_dw)
rho_dw_0 = np.sum(np.abs(solver_dw.get_psi())**2, axis=1).real
solver_dw.run_simulation()
rho_dw_T = np.sum(np.abs(solver_dw.get_psi())**2, axis=1).real

fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.plot(x, rho_dw_0, 'k--', alpha=0.6, label='Densidad Inicial (t = 0)')
ax1.plot(x, rho_dw_T, 'b-', lw=2, label='Densidad Final (t = 3.5)')
ax1.set_xlabel('Posición x')
ax1.set_ylabel(r'Densidad de Probabilidad $\rho(x)$', color='b')

ax2 = ax1.twinx()
ax2.plot(x, [pot_double_well.evaluate([xi]) for xi in x], 'r-', alpha=0.4, lw=2, label='Potencial Doble Pozo V(x)')
ax2.set_ylabel('Potencial V(x)', color='r')

ax1.set_title('Tunelamiento e Interferencia en un Doble Pozo Cuántico')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. Efecto Klein (Tunelamiento Relativista a través de una Barrera Supercrítica)

Consideremos un paquete gaussiano con momento $p_0 = 3.0$ y masa $m = 1.0$, cuya energía cinética relativista total es:

$$E = \sqrt{p_0^2 + m^2} = \sqrt{10} \approx 3.162$$

Colocamos una barrera rectangular en $x \in [1.0, 5.0]$ de altura supercrítica $V_0 = 6.0 > E + m \approx 4.162$. En la ecuación de Schrödinger no relativista la transmisión a través de una barrera tan alta y ancha sería exponencialmente nula; sin embargo, en la ecuación de Dirac el paquete se transmite con amplitud apreciable a través de la zona $V_0 > E + m$.

In [ ]:
p0_klein = 3.0
E_klein = np.sqrt(p0_klein**2 + electron_mass**2)
V0_klein = 6.0  # Régimen de Klein: V0 > E + m
barrier_left, barrier_right = 1.0, 5.0

print(f"Energía del paquete E = {E_klein:.3f}, Umbral de Klein E + m = {E_klein + electron_mass:.3f}, Barrera V0 = {V0_klein:.3f}")

def klein_barrier_func(r):
    pos_x = r[0]
    if barrier_left <= pos_x <= barrier_right:
        return V0_klein
    return 0.0

pot_klein = ScalarPotential(klein_barrier_func)
spinor_klein = ConstantSpinor(momentum=[p0_klein, 0.0, 0.0])
initial_klein = GaussianPacket(spinor_klein, center=[-4.0, 0.0, 0.0], spatial_width=0.9)

problem_klein = (
    DiracProblemBuilder()
    .set_grid(grid_1d)
    .set_initial_state(initial_klein)
    .set_potential(pot_klein)
    .set_boundary_condition(_core.AbsorbingBoundary(strength=0.3, width=25))
    .set_time_parameters(time_step=0.004, total_time=4.5)
    .build()
)

solver_klein = DiracSolver(problem_klein)
rho_k_init = np.sum(np.abs(solver_klein.get_psi())**2, axis=1).real
norm_init = np.sum(rho_k_init) * dV

solver_klein.run_simulation()
rho_k_final = np.sum(np.abs(solver_klein.get_psi())**2, axis=1).real

# Cálculo de probabilidades reflejada, dentro de la barrera y transmitida (normalizadas)
mask_R = x < barrier_left
mask_B = (x >= barrier_left) & (x <= barrier_right)
mask_T = x > barrier_right

prob_R = np.sum(rho_k_final[mask_R]) * dV / norm_init
prob_B = np.sum(rho_k_final[mask_B]) * dV / norm_init
prob_T = np.sum(rho_k_final[mask_T]) * dV / norm_init

print(f"Fracción Reflejada   (R): {prob_R:.4f}")
print(f"Fracción en Barrera  (B): {prob_B:.4f}")
print(f"Fracción Transmitida (T): {prob_T:.4f}  <-- ¡Tunelamiento de Klein!")

In [ ]:
# Visualización del Efecto Klein
fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.plot(x, rho_k_init / norm_init, 'k--', alpha=0.6, label='Paquete Incidente (t = 0)')
ax1.plot(x, rho_k_final / norm_init, 'b-', lw=2, label='Estado Dispersado (t = 4.5)')
ax1.axvspan(barrier_left, barrier_right, color='crimson', alpha=0.18, label=f'Barrera de Klein ($V_0 = {V0_klein} > E+m$)')
ax1.set_xlabel('Posición x')
ax1.set_ylabel('Densidad de Probabilidad Normalizada')
ax1.set_title(f'Efecto Klein en 1D: R = {prob_R:.2%}, B = {prob_B:.2%}, T = {prob_T:.2%}')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()